# 🇹🇭 Thai NLP Toolkit — Multi-Account Training on Colab

Notebook นี้ออกแบบมาเป็นพิเศษสำหรับกรณีที่**ต้องสลับ Google Account ไปมาหลายรอบ** เมื่อสิทธิ์การใช้งาน GPU (quota) ของบัญชีแรกหมดลง

### 💡 แนวคิดหลักในการสลับ Account:
1. **เก็บ Tokenizer ถาวร**: เมื่อเทรน Tokenizer (Step 5a + 5b) เสร็จใน Account แรกแล้ว เราจะก๊อปปี้ไปสำรองใน Google Drive
2. **ไม่ต้องเทรนใหม่**: เมื่อสลับบัญชีใหม่ ให้ข้ามการเทรน Tokenizer แล้วสั่งดึงไฟล์จาก Google Drive แทนเพื่อประหยัดเวลา
3. **เทรนโมเดลต่อจากเดิม (Resume)**: บันทึก Checkpoint โมเดลไว้ใน Google Drive เพื่อโหลดมาเทรนต่อจากขั้นตอนเดิมได้ทันที

---
> ⚠️ **ก่อนเริ่ม**: ไปที่เมนูด้านบน **Runtime (รันไทม์) → Change runtime type (เปลี่ยนประเภทการรันไทม์) → GPU (T4)**

## 1. เชื่อมต่อ Google Drive (ขั้นตอนแรกสุด 📂)

ต้องเชื่อมต่อ Google Drive ก่อนเป็นอันดับแรก เพื่อดึงข้อมูลสำรองหรือ Checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์และตัวสำรองข้อมูลบน Google Drive ของคุณ
!mkdir -p /content/drive/MyDrive/thai_nlp_tokenizer_backup
!mkdir -p /content/drive/MyDrive/thai_nlp_outputs

## 2. ตรวจสอบ GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Clone โปรเจคจาก GitHub และติดตั้ง Dependencies

In [ ]:
%cd /content
import os, shutil
if os.path.exists("thai-nlp-toolkit"):
    shutil.rmtree("thai-nlp-toolkit")

!git clone https://github.com/puttibenz/thai-nlp-toolkit.git
%cd thai-nlp-toolkit

# ติดตั้ง Library ที่จำเป็น
!pip install -q sentencepiece>=0.1.99 pythainlp>=4.0.0 datasets>=2.14.0 pyyaml>=6.0 tqdm>=4.65.0 scikit-learn>=1.3.0

## 4. ดาวน์โหลด Raw Datasets (สำหรับเทรนโมเดล)

In [ ]:
!python -m data.download --task all

## 5. จัดการ Tokenizer (เลือกเพียงข้อเดียว: 5A หรือ 5B) 🛠️

### 📌 [ทางเลือก 5A]: เทรน Tokenizer ใหม่และสำรองข้อมูล (เฉพาะการรันครั้งแรกสุด)

รันข้อนี้เฉพาะใน Account แรกที่คุณเริ่มโปรเจกต์เท่านั้น เมื่อรันเสร็จระบบจะก๊อปปี้ไปสำรองไว้ใน Google Drive อัตโนมัติ

In [ ]:
# 1. ดาวน์โหลดคอร์ปัสภาษาไทยเพื่อใช้เทรน
!python -m data.download_corpus --max_lines 2000000

# 2. เทรน Tokenizer (จำกัด 1M บรรทัด ป้องกัน OOM บน Colab)
!python -m tokenizer.train_tokenizer \
    --corpus data/corpus/thai_corpus.txt \
    --output_dir tokenizer \
    --vocab_size 32000

# 3. คัดลอกไฟล์ Tokenizer ไปสำรองไว้บน Google Drive
!cp -r /content/thai-nlp-toolkit/tokenizer/thai_bpe.* /content/drive/MyDrive/thai_nlp_tokenizer_backup/
!cp /content/thai-nlp-toolkit/tokenizer/tokenizer_config.json /content/drive/MyDrive/thai_nlp_tokenizer_backup/
print("\n✅ เทรนและทำการสำรองข้อมูล Tokenizer ลง Google Drive เรียบร้อยแล้ว!")

### 📌 [ทางเลือก 5B]: กู้คืน Tokenizer จาก Google Drive (สำหรับเมื่อเปลี่ยน Account ใหม่)

หากคุณเปลี่ยนมาใช้ Account ถัดไป ให้**ข้ามข้อ 5A** แล้วมารันข้อนี้ได้เลยเพื่อดึงคลังคำศัพท์เดิมมาใช้ทันที

In [ ]:
# ดึงไฟล์ประกอบทั้งหมดมาลงในโปรเจคโดยไม่ให้เกิดการซ้อนโฟลเดอร์
!cp -r /content/drive/MyDrive/thai_nlp_tokenizer_backup/thai_bpe.* /content/thai-nlp-toolkit/tokenizer/
!cp /content/drive/MyDrive/thai_nlp_tokenizer_backup/tokenizer_config.json /content/thai-nlp-toolkit/tokenizer/
print("✅ ดึงไฟล์ Tokenizer จาก Google Drive สำเร็จ!")

## 6. ตรวจสอบความพร้อมของระบบก่อนเริ่มเทรน 📊

In [ ]:
import os
import sys
sys.path.append(os.path.abspath("."))

required_files = [
    "tokenizer/thai_bpe.model",
    "tokenizer/thai_bpe.vocab",
    "tokenizer/tokenizer_config.json",
    "data/raw/ner_train.jsonl",
    "data/raw/sent_train.tsv",
    "data/raw/qa_train.json",
]

for f in required_files:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"{status}  {f}")

# ตรวจสอบการโหลด Tokenizer จริง
from tokenizer.thai_tokenizer import ThaiTokenizer
tok = ThaiTokenizer.from_pretrained("tokenizer")
print(f"\n📊 Tokenizer vocab size: {tok.vocab_size:,}")
assert tok.vocab_size >= 30000, f"❌ vocab_size ยังเล็กเกินไป ({tok.vocab_size}) — ตรวจสอบการทำขั้นตอนที่ 5 ใหม่"
print("✅ Tokenizer พร้อมใช้งาน!")

## 7. เริ่มต้นเทรนโมเดล (เลือกเพียงข้อเดียวตามสถานะ) 🚀

### 📌 [ทางเลือก 7A]: เริ่มเทรนจาก 0 (สำหรับเริ่มสเตปแรกสุด)

In [ ]:
# เริ่มเทรนและส่งไฟล์โมเดลไปเก็บที่ Google Drive โดยตรง
!python -m scripts.train --device cuda --output_dir /content/drive/MyDrive/thai_nlp_outputs

### 📌 [ทางเลือก 7B]: โหลดเซฟมาเทรนต่อ (สำหรับใช้เมื่อเปลี่ยน Account ใหม่)

รันบรรดทัดด้านล่างเพื่อดึงค่า Checkpoint ตัวเดิมจาก Google Drive มาเทรนต่อจากขั้นตอนเดิมทันที

In [ ]:
# สั่งเทรนต่อจาก checkpoint ล่าสุดที่เก็บไว้ใน Drive
!python -m scripts.train --device cuda \
    --output_dir /content/drive/MyDrive/thai_nlp_outputs \
    --resume /content/drive/MyDrive/thai_nlp_outputs/checkpoint_best

## 8. จัดเตรียมทรัพยากรสำหรับประเมินผล

In [ ]:
# คัดลอก config และ tokenizer ไปเก็บร่วมกับ checkpoint บน Google Drive เพื่อความครบถ้วน
!cp configs/base_config.yaml /content/drive/MyDrive/thai_nlp_outputs/config.yaml
!cp -r tokenizer /content/drive/MyDrive/thai_nlp_outputs/
print("✅ ย้ายเอกสารประกอบเสร็จสิ้น พร้อมนำไปใช้งาน")

## 9. ประเมินผลโมเดล (Evaluation)

In [ ]:
# ประเมินคะแนน F1 / Accuracy ของแต่ละ Task บน Test sets
!python -m scripts.evaluate --model_dir /content/drive/MyDrive/thai_nlp_outputs --task all

## 10. ทดลองทดสอบใช้งานจริง (Inference Test)

In [ ]:
from inference.pipeline import ThaiNLPPipeline

# โหลดโมเดลที่เราพึ่งเทรนเสร็จ
pipeline = ThaiNLPPipeline(model_dir="/content/drive/MyDrive/thai_nlp_outputs")

# 1. ทดสอบ NER และ Sentiment Analysis
text = "สมชายชอบไปเที่ยวที่เชียงใหม่แต่เมื่อวานเขารู้สึกเศร้ามาก"
res = pipeline.predict(text, tasks=["ner", "sentiment"])
print("=== ผลการวิเคราะห์ ===")
print(f"ข้อความ: {text}")
print("NER:", res["ner"])
print("Sentiment:", res["sentiment"])
print()

# 2. ทดสอบ QA (Question Answering)
context = "ประเทศไทยแบ่งเขตการปกครองออกเป็น 76 จังหวัด และมีกรุงเทพมหานครเป็นเขตปกครองพิเศษ"
question = "ประเทศไทยมีกี่จังหวัด?"
res_qa = pipeline.predict(text="", tasks=["qa"], question=question, context=context)
print("=== ผลการตอบคำถาม ===")
print(f"คำถาม: {question}")
print(f"บริบท: {context}")
print("คำตอบจากโมเดล:", res_qa["qa"]["answer"])